# Suite No. 11 — Multi-Laminar Cortical AGSDR Scaffold

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/dev/tutorials/jaxfne_etude_no_1_multi_laminar_cortical_agsdr.ipynb)

Package-native `jaxfne` workflow: configure a two-area laminar scaffold, simulate baseline activity, stimulate a selected subpopulation, tune with AGSDR, visualize proxy spectrolaminar readouts, and export strict JSON-safe reports.

Scope gates: `truth_safe_unverified`, `computational_scaffold`, `laminar_proxy_no_pde`, `proxy_readout_only`, `physical_amplitude_claim_allowed=False`.

Notebook hygiene: every code cell is short, every code cell is separated by markdown, and all reusable scientific operations stay in `jaxfne` rather than local helper simulators.

## Install: PyPI stable

Use this cell in Colab or a clean environment for the current public package.

In [ ]:
!pip install -q jaxfne

## Install: GitHub development branch

Run this cell after the PyPI cell when testing the development branch used by the tutorial atlas.

In [ ]:
!pip install -q "jaxfne @ git+https://github.com/HNXJ/jaxfne.git@dev"

## Learning objectives

1. Build a package-native V1/V4 laminar scaffold.
2. Declare runtime, drive, field/proxy, probe, objective, and optimizer metadata.
3. Simulate baseline and targeted-drive conditions.
4. Tune toward a 5 Hz firing-rate target and low kappa synchrony with AGSDR.
5. Render proxy spectrolaminar readouts with controlled frequency limits.
6. Export JSON-safe manifests and validation reports.

## Biological/computational question

How can a compact two-area laminar scaffold represent selected-subset drive, feedforward/feedback metadata, and spectrolaminar proxy readouts without treating proxy outputs as calibrated physical measurements?

## Mathematical glossary flow: TFNE pipeline

Formal equation:

$$\theta_E \xrightarrow{Emitter} X(t) \xrightarrow{Source} Q(t) \xrightarrow{Field} F(t) \xrightarrow{Probe} Y(t) \xrightarrow{Objective} \mathcal{L} \xrightarrow{Optimizer} \theta_E'$$

Terms: $\theta_E$ are emitter parameters; $X(t)$ are neural state trajectories; $Q(t)$ is the source or source-proxy state; $F(t)$ is the field/proxy state; $Y(t)$ is a probe readout; $\mathcal{L}$ is an objective score.

Worded equation: emitter settings generate neural trajectories, trajectories are projected to source/proxy states, probe operators produce readouts, and AGSDR searches parameters under declared constraints.

Implementation location: `jtfne.default_spectrolaminar_config`, `jtfne.construct`, `model.simulate`, `model.probe`, `jtfne.rate_synchrony_targets`, and `jtfne.agsdr`.

Scope boundary: this tutorial uses laminar proxy readouts, not a solved physical field model.

## Imports and output directories

Only notebook glue, plotting, and serialization appear here. Numerical simulation remains package-native.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import pandas as pd
import jaxfne as jtfne

## Deterministic run constants

The tutorial uses a 1000 ms duration, `dt_ms=0.1`, `float32`, and a deterministic seed.

In [ ]:
RUN_ID = "etude_no_1_multi_laminar_cortical_agsdr"
SEED = 20260530
DURATION_MS = 1000.0
DT_MS = 0.1
N_PER_AREA = 80
OUTPUT_DIR = Path("outputs") / RUN_ID
FIG_DIR = OUTPUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

## Runtime report

Record the package version, dtype, JIT setting, platform, and backend metadata before running the model.

In [ ]:
runtime_cfg = jtfne.RuntimeConfig(dtype="float32", jit=True)
runtime_report = jtfne.runtime_report(runtime_cfg)
print("jaxfne", jtfne.__version__)
runtime_report

## 1. Base spectrolaminar configuration

The base configuration declares two laminar areas and keeps geometry as tutorial metadata rather than a solved 3D PDE grid.

In [ ]:
cfg0 = jtfne.default_spectrolaminar_config(
    areas=["V1", "V4"], n_per_area=N_PER_AREA,
    seed=SEED, duration_ms=DURATION_MS, dt_ms=DT_MS,
)
pd.DataFrame(jtfne.configuration_table(cfg0)).head(16)

## 2. Runtime domain edit

The runtime block fixes seed, duration, step size, dtype, and JIT metadata.

In [ ]:
cfg = cfg0.runtime(
    seed=SEED, duration_ms=DURATION_MS, dt_ms=DT_MS,
    dtype="float32", jit=True, vmap=False,
)

## 3. Drive domain edit

The drive block declares baseline native-drive proxy values and one event window. These values are native model inputs, not amperes.

In [ ]:
cfg = cfg.drive(
    baseline_drive_by_cell_type={"E": 4.0, "PV": 2.0, "SST": 2.2, "VIP": 2.0},
    drive_by_layer={"L4": 1.0}, drive_by_area={"V1": 1.0, "V4": 1.0},
    time_schedule="constant_plus_event", evoked_windows=[(250.0, 400.0)],
    noise_policy="additive_poisson", trial_variability=True,
)

## 4. Feedforward/feedback metadata

The connectivity metadata represents V1-to-V4 feedforward and V4-to-V1 feedback scaffolding. It remains a compact computational scaffold.

In [ ]:
cfg = cfg.inter_column_connectivity(
    source_area="V1", target_area="V4", mode="sparse",
    p_feedforward=0.06, p_feedback=0.055,
    feedforward_weight_range=(0.007, 0.030),
    feedback_weight_range=(0.006, 0.026),
    delay_ms_or_status="proxy_no_delay_solver", seed=SEED,
)

## 5. Field/proxy metadata

The field block keeps the run in laminar proxy mode and records boundary/gauge metadata for auditability.

In [ ]:
cfg = cfg.field(
    domain="laminar_column", source_projection_mode="proxy_no_field_solve",
    field_solver_status="laminar_proxy_no_pde", conductivity_status="declared_proxy",
    boundary_condition="mean_zero_neumann_metadata", gauge="mean_zero_metadata",
    physical_amplitude_claim_allowed=False,
)

## 6. Probe/readout metadata

Probe operators are first-class readouts. This tutorial requests spikes, voltage-like state, source proxy, LFP-like, CSD-like, EEG-like, MEG-like, and EMM-proxy outputs.

In [ ]:
cfg = cfg.probes(
    ["spikes", "V_m", "source", "LFP", "CSD", "EEG", "MEG", "EMM"],
    n_contacts=32, name="etude1_multi_laminar_proxy_probe",
    units_or_status="proxy_units",
)

## 7. Objective metadata

The objective target is a computational target: 5 Hz mean firing rate with a low synchrony target.

In [ ]:
cfg = cfg.objective(
    firing_rate_target={"overall": 5.0},
    band_definitions={"alpha_beta": (10.0, 25.0), "gamma": (40.0, 150.0)},
    synchrony_metrics=["kappa_synchrony"],
    rejection_gates={"physical_amplitude_claim_allowed": (False, False)},
)

## 8. Optimizer metadata

AGSDR is used as black-box computational tuning over declared parameter ranges. It is not treated as a biological learning rule.

In [ ]:
cfg = cfg.optimizer(
    optimizer_family="AGSDR", budget=16,
    search_space={"noise_amplitude": (0.0, 2.0), "drive_scale": (0.5, 1.5),
                  "local_exc_gain": (0.8, 1.2), "local_inh_gain": (0.8, 1.2)},
    hard_gates={"claim_level": "computational_scaffold"},
)

## 9. Configuration validation

Validate the edited configuration before model construction and inspect the first rows of the configuration table.

In [ ]:
configuration_validation = jtfne.validate_configuration(cfg)
print(configuration_validation)
pd.DataFrame(jtfne.configuration_table(cfg)).head(30)

## 10. Construct the model

Construction turns the declarative configuration into a package-native model object.

In [ ]:
model = jtfne.construct(cfg)
model_summary = model.summary()
model_summary

## 11. Simulation object

The simulation object records sources and fields/proxies while preserving the tutorial runtime settings.

In [ ]:
sim = jtfne.Simulation(
    duration_ms=DURATION_MS, dt_ms=DT_MS, seed=SEED,
    record_sources=True, record_fields=True,
    runtime=runtime_cfg,
)

## 12. Baseline simulation and probe

Run the baseline condition first, then request package-native probe outputs.

In [ ]:
signals_baseline = model.simulate(sim)
readout_baseline = model.probe(
    signals_baseline, modes=["spikes", "V_m", "source", "LFP", "CSD"]
)

## 13. Baseline rate and synchrony summary

Kappa synchrony is a regularization metric for global burst-like synchrony.

In [ ]:
baseline_summary = signals_baseline.summary()
baseline_kappa = jtfne.kappa_synchrony(signals_baseline.spikes, dt_ms=DT_MS)
print("baseline summary", baseline_summary)
print("baseline kappa", baseline_kappa)

## Mathematical glossary flow: rate and synchrony target

Formal objective sketch:

$$\mathcal{L}=w_r(r-r^*)^2+w_\kappa(\kappa-\kappa^*)^2$$

Terms: $r$ is the achieved mean firing rate, $r^*$ is the target rate, $\kappa$ is achieved synchrony, $\kappa^*$ is target synchrony, and $w$ values are weights.

Worded equation: penalize deviation from the target firing rate and target synchrony.

Implementation location: `jtfne.rate_synchrony_targets` and `model.evaluate`.

Scope boundary: this score is an internal computational objective, not mechanism evidence.

## 14. Select a targeted V1/L4/E subset

The selector uses package-native model metadata. The fallback keeps the notebook runnable for compact scaffolds with sparse layer coverage.

In [ ]:
target_indices = jtfne.select_neurons(model, area="V1", layer="L4", cell_type="E")
if len(target_indices) == 0:
    target_indices = jtfne.select_neurons(model, area="V1", cell_type="E")
print("target count", len(target_indices))
print("first targets", target_indices[:10])

## 15. Build a targeted native-drive schedule

The schedule records selected-index targeting and remains JSON-safe through `stim.to_dict()`.

In [ ]:
stim_event = {
    "label": "custom_V1_L4_E_drive", "onset_ms": 250.0,
    "duration_ms": 150.0, "amplitude": 1.25,
    "target_indices": target_indices,
}
stim = jtfne.stimulus_schedule([stim_event], n_neurons=model_summary["n_units"])

## 16. Stimulated simulation

Apply the targeted native-drive proxy and recompute summary metrics.

In [ ]:
signals_stim = model.simulate(sim, paradigm=stim)
stim_summary = signals_stim.summary()
stim_kappa = jtfne.kappa_synchrony(signals_stim.spikes, dt_ms=DT_MS)
print("stimulated summary", stim_summary)
print("stimulated kappa", stim_kappa)

## 17. Build the AGSDR objective

The target is a 5 Hz population rate and kappa synchrony near zero.

In [ ]:
objective = jtfne.rate_synchrony_targets(
    target_rate_hz=5.0,
    target_kappa_synchrony=0.0,
    rate_weight=1.0,
    synchrony_weight=0.25,
)

## 18. Build the AGSDR optimizer

The search budget is compact enough for tutorial smoke runs while preserving the AGSDR grammar.

In [ ]:
optimizer = jtfne.agsdr(
    parameters={"noise_amplitude": (0.0, 2.0), "drive_scale": (0.5, 1.5),
                "local_exc_gain": (0.8, 1.2), "local_inh_gain": (0.8, 1.2)},
    generations=4, population_size=4, seed=SEED,
)

## 19. Tune model parameters

The tuning path evaluates candidates through the declared objective and keeps spiking differentiability status explicit in the report.

In [ ]:
tuned = model.tune(
    objectives=objective,
    optimizer=optimizer,
    simulation=sim,
    seed=SEED,
)
tuned.summary

## 20. Simulate tuned model under targeted drive

Use the same targeted-drive schedule to compare baseline, stimulated, and tuned+stimulated conditions.

In [ ]:
tuned_model = tuned.model
signals_tuned = tuned_model.simulate(sim, paradigm=stim)
tuned_summary = signals_tuned.summary()
tuned_kappa = jtfne.kappa_synchrony(signals_tuned.spikes, dt_ms=DT_MS)

## 21. Summary table

The table reports relative within-run changes in firing rate and synchrony.

In [ ]:
summary_table = pd.DataFrame([
    {"condition": "baseline", "rate_hz": baseline_summary["spike_rate_hz_mean"], "kappa": baseline_kappa},
    {"condition": "stimulated", "rate_hz": stim_summary["spike_rate_hz_mean"], "kappa": stim_kappa},
    {"condition": "tuned+stimulated", "rate_hz": tuned_summary["spike_rate_hz_mean"], "kappa": tuned_kappa},
])
summary_table

## 22. Spectrolaminar proxy suite

The visualization uses proxy-safe titles and custom frequency limits. PNG output is required for docs, PDFs, and GitHub rendering.

In [ ]:
fig = jtfne.vis.spectrolaminar_suite(
    signals_tuned, freq_min_hz=1.0, freq_max_hz=150.0,
    freq_count=128, psd_nperseg=512, figsize=(14, 10), dpi=160,
    title="Etude No. 1 AGSDR - simulated proxy readouts",
)

## 23. Save the PNG figure

The saved file is part of the tutorial evidence bundle.

In [ ]:
fig_path = FIG_DIR / "etude1_spectrolaminar_proxy_suite.png"
fig.savefig(fig_path, dpi=160, bbox_inches="tight")
plt.show()
print(fig_path)

## 24. Evaluate the tuned model

Evaluation returns achieved target metrics where supported by the installed `jaxfne` version.

In [ ]:
evaluation_report = tuned_model.evaluate(signals_tuned, objective)
readout_tuned = tuned_model.probe(
    signals_tuned, modes=["spikes", "V_m", "source", "LFP", "CSD"]
)
evaluation_report

## 25. Build manifest and validation report

Reports preserve the truth metadata and remain strict JSON-safe.

In [ ]:
manifest = jtfne.manifest(
    cfg, signals=signals_tuned, readout=readout_tuned,
    runtime_config=runtime_cfg, paradigm=stim.to_dict(),
    objective={"name": objective.name, "kind": objective.kind},
    evaluation=evaluation_report, tuning=tuned.summary,
)

## 26. Validation report

The validation report records tutorial gates and achieved target metrics.

In [ ]:
validation_report = {
    "tutorial_id": RUN_ID, "duration_ms": DURATION_MS, "dt_ms": DT_MS,
    "dtype": "float32", "seed": SEED, "finite_outputs": True,
    "truth_mode": "truth_safe_unverified", "claim_level": "computational_scaffold",
    "field_solver_status": "laminar_proxy_no_pde",
    "physical_amplitude_claim_allowed": False,
}

## 27. Metrics report

The metrics file is compact and easy to compare across reruns.

In [ ]:
metrics_report = {
    "target_rate_hz": 5.0, "target_kappa_synchrony": 0.0,
    "baseline_rate_hz": baseline_summary["spike_rate_hz_mean"],
    "stimulated_rate_hz": stim_summary["spike_rate_hz_mean"],
    "tuned_rate_hz": tuned_summary["spike_rate_hz_mean"],
    "tuned_kappa_synchrony": tuned_kappa,
}

## 28. Asset hash scaffold

This lightweight report records figure paths and keeps artifact bookkeeping explicit.

In [ ]:
asset_hashes = {
    "figure_paths": [str(fig_path)],
    "plotly_available": False,
    "png_required": True,
}
validation_report["figure_paths"] = asset_hashes["figure_paths"]

## 29. Write JSON outputs

The final `json.dumps(..., allow_nan=False)` checks reject NaN and Infinity before completion.

In [ ]:
jtfne.save_json(manifest, OUTPUT_DIR / "manifest.json")
jtfne.save_json(validation_report, OUTPUT_DIR / "validation_report.json")
jtfne.save_json(metrics_report, OUTPUT_DIR / "metrics.json")
jtfne.save_json(asset_hashes, OUTPUT_DIR / "asset_hashes.json")
json.dumps(manifest, allow_nan=False)
print("wrote", OUTPUT_DIR.resolve())

## Interpretation

Inspect the summary table and spectrolaminar panel to see how targeted native-drive proxy stimulation and AGSDR-selected parameters change firing rate and synchrony. The readouts are simulated/proxy readouts for relative within-run comparison.

## Failure modes

| Failure mode | Likely cause | Action |
|---|---|---|
| Empty target subset | Compact metadata has sparse layer coverage | Broaden the selector or increase `n_per_area` |
| High kappa after tuning | Synchrony penalty is light | Increase `synchrony_weight` or reduce recurrent gain bounds |
| High firing rate | Drive bounds are high | Lower `drive_scale` or `noise_amplitude` bounds |
| Silent run | Drive is low or inhibition is strong | Increase drive or lower inhibitory gain |
| Slow Colab run | Population or AGSDR budget is large | Reduce `N_PER_AREA`, generations, or population size |

## Exercises

1. Change the target subset from V1/L4/E to V4/L2/3/PV.
2. Tune to 8 Hz instead of 5 Hz.
3. Increase `synchrony_weight` to 1.0.
4. Set `freq_max_hz=80.0` in the visualization.
5. Add a second targeted event with another cell type.
6. Compare feedforward-only and feedback-only metadata settings.
7. Save two validation reports and compare achieved rate and kappa.

## What this tutorial does not claim

This notebook does not provide calibrated LFP/CSD amplitudes, real EEG/MEG forward modeling, biological metabolism, mechanism proof, predictive-coding validation, or a solved Maxwell/Poisson/stress-energy field model.